# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

[https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json](https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# Retrieve record set @id's from dataset metadata
record_sets = getattr(metadata, 'recordSet', []) if hasattr(metadata, 'recordSet') else []

if len(record_sets) == 0:
    # Retrieve from Croissant JSON in case Python data model doesn't have 'recordSet' populated
    import requests
    schema_json = requests.get(croissant_url).json()
    record_sets = schema_json.get('recordSet', [])

if len(record_sets) == 0:
    print("No record sets found in the dataset metadata.")
else:
    print(f"Available record sets by @id:")
    for rs in record_sets:
        rs_id = rs['@id'] if isinstance(rs, dict) else rs
        print(f"- {rs_id}")

    # Show fields for each record set by @id
    for rs in record_sets:
        rs_id = rs['@id'] if isinstance(rs, dict) else rs
        rs_obj = dataset.record_set(rs_id)
        print(f"\nFields for record set {rs_id}:")
        for field in rs_obj.fields:
            print(f"  - {getattr(field, '@id', None)}: {getattr(field, 'name', '')}")

## 3. Data Extraction
Load data from specific record set(s) into DataFrames for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# List all available record set @id's for extraction
record_set_ids = []
for rs in record_sets:
    rs_id = rs['@id'] if isinstance(rs, dict) else rs
    record_set_ids.append(rs_id)

# Extract data from each record set
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if len(records) > 0:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df

for rs_id in dataframes:
    print(f"Columns for record set {rs_id}: {dataframes[rs_id].columns.tolist()}")
    display(dataframes[rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare the dataset.

In [ ]:
# Choose one DataFrame and field ID for numeric analysis
example_record_set_id = None
example_numeric_field_id = None
example_group_field_id = None
# Find a record set with numeric fields
for rs_id, df in dataframes.items():
    for col in df.columns:
        # Use columns with string 'age' or likely hormone value as demo
        if 'age' in col.lower() or 'level' in col.lower():
            example_record_set_id = rs_id
            example_numeric_field_id = col
            break
    if example_record_set_id is not None:
        break
# Try to find a groupable field
if example_record_set_id is not None:
    df = dataframes[example_record_set_id]
    for col in df.columns:
        if col.lower() in ['phenotype', 'patient_id', 'id', 'zygocity', 'pathogenicity']:
            example_group_field_id = col
            break

if example_record_set_id is not None and example_numeric_field_id is not None:
    print(f"Performing EDA on record set {example_record_set_id}, field {example_numeric_field_id}")
    # Remove missing/invalid values
    numeric_df = df[pd.to_numeric(df[example_numeric_field_id], errors='coerce').notnull()].copy()
    numeric_df[example_numeric_field_id] = pd.to_numeric(numeric_df[example_numeric_field_id], errors='coerce')
    # Filter for values > threshold
    threshold = numeric_df[example_numeric_field_id].mean()
    filtered_df = numeric_df[numeric_df[example_numeric_field_id] > threshold]
    print(f"Filtered records with {example_numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize
    filtered_df[f"{example_numeric_field_id}_normalized"] = (filtered_df[example_numeric_field_id] - filtered_df[example_numeric_field_id].mean()) / filtered_df[example_numeric_field_id].std()
    print(f"Normalized {example_numeric_field_id} for filtered records:")
    display(filtered_df[[example_numeric_field_id, f"{example_numeric_field_id}_normalized"]].head())

    # Group if possible
    if example_group_field_id is not None and example_group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(example_group_field_id).mean(numeric_only=True)
        print(f"Grouped data by {example_group_field_id}:")
        display(grouped_df.head())
else:
    print("No suitable numeric fields found for EDA in available record sets.")

## 5. Visualization
Visualize distributions or relationships between fields in the dataset using matplotlib or seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if example_record_set_id is not None and example_numeric_field_id is not None:
    plt.figure(figsize=(6,4))
    sns.histplot(numeric_df[example_numeric_field_id], bins=5, kde=True)
    plt.title(f"Distribution of {example_numeric_field_id} in {example_record_set_id}")
    plt.xlabel(example_numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    if example_group_field_id is not None and example_group_field_id in numeric_df.columns:
        plt.figure(figsize=(6,4))
        sns.boxplot(x=numeric_df[example_group_field_id], y=numeric_df[example_numeric_field_id])
        plt.title(f"{example_numeric_field_id} by {example_group_field_id}")
        plt.xlabel(example_group_field_id)
        plt.ylabel(example_numeric_field_id)
        plt.show()
else:
    print("Visualization skipped: No suitable numeric fields loaded.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Explored record sets, fields, and their unique `@id`s using Croissant schema and `mlcroissant`.
- Demonstrated record extraction, filtering, normalization, and grouping using dynamic field and record set IDs.
- Visualizations highlighted variable distributions and relationships for further clinical or genomic insight.
- The dataset's small sample size underscores caution for generalizing results, but it provides a template for clinical data exploration in rare disease research.